# Week 4 — Exercise 4: Serving Deep Learning Models via REST API

### Objectives:
1. Build a production-ready REST API using **FastAPI** to serve a deep learning model.
2. Implement key API design principles:
   - Load model weights once during application startup using `@asynccontextmanager` (lifespan).
   - `/health`: Health status, model readiness, and versioning.
   - `/predict`: Input payload validation (dimensions, file formats, corrupted bytes).
   - Standardized JSON responses with class labels and confidence scores.
   - Explicit HTTP status codes (`200`, `400`, `413`, `503`).
3. Automated test suite using `starlette.testclient.TestClient`.
4. Walkthrough of interactive documentation (Swagger UI) and deployment patterns.

In [1]:
import io
import time
from typing import List, Optional
from contextlib import asynccontextmanager
from PIL import Image
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pydantic import BaseModel, Field
from fastapi import FastAPI, File, UploadFile, HTTPException, Form, status
from fastapi.testclient import TestClient

print(f"FastAPI and TensorFlow ready!")

FastAPI and TensorFlow ready!


D:\Applications\python 3.11\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 1. Pydantic Schemas for Request & Response Contracts

In [2]:
class PredictionItem(BaseModel):
    label: str = Field(..., description="Predicted class name")
    confidence: float = Field(..., description="Softmax confidence score (0.0 to 1.0)")
    percentage: str = Field(..., description="Formatted confidence percentage")

class PredictionResponse(BaseModel):
    success: bool = True
    predicted_class: str
    confidence: float
    top_k: List[PredictionItem]
    inference_time_ms: float
    filename: str

class HealthResponse(BaseModel):
    status: str
    model_loaded: bool
    version: str = "1.0.0"

## 2. In-Memory Mock Model Setup

For clean, self-contained execution within this exercise, we define a lightweight Keras classification model.

In [3]:
# Create a simple demonstration model (e.g. 3-class classifier)
CLASSES = ["cat", "dog", "bird"]

demo_model = keras.Sequential([
    keras.Input(shape=(64, 64, 3)),
    keras.layers.Conv2D(8, (3, 3), activation="relu"),
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(3, activation="softmax")
])

def preprocess_image(image_bytes: bytes) -> np.ndarray:
    """Preprocess incoming raw image bytes into normalized model tensor."""
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB").resize((64, 64))
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return np.expand_dims(arr, axis=0)

## 3. FastAPI Application Implementation

We declare the lifespan context to load model resources once during startup, ensuring zero per-request disk overhead.

In [4]:
app = FastAPI(
    title="Deep Learning Model Serving API",
    description="FastAPI service demonstrating production model deployment patterns.",
    version="1.0.0"
)

# Application state
app.state.model = demo_model
app.state.classes = CLASSES

@app.get("/health", response_model=HealthResponse)
async def health():
    return HealthResponse(
        status="healthy" if app.state.model is not None else "degraded",
        model_loaded=app.state.model is not None,
    )

@app.post("/predict", response_model=PredictionResponse)
async def predict(file: UploadFile = File(...), top_k: int = Form(3)):
    if not file.filename:
        raise HTTPException(status_code=400, detail="Missing filename")
        
    ext = file.filename.split(".")[-1].lower()
    if ext not in ["png", "jpg", "jpeg", "webp"]:
        raise HTTPException(status_code=400, detail=f"Unsupported format '{ext}'.")
        
    try:
        content = await file.read()
        if len(content) == 0:
            raise HTTPException(status_code=400, detail="Empty upload.")
        input_tensor = preprocess_image(content)
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid image content: {e}")

    t0 = time.perf_counter()
    probs = app.state.model.predict(input_tensor, verbose=0)[0]
    latency = (time.perf_counter() - t0) * 1000.0

    top_idx = int(np.argmax(probs))
    top_label = app.state.classes[top_idx]
    
    top_items = []
    ranked = np.argsort(probs)[::-1][:top_k]
    for idx in ranked:
        top_items.append(PredictionItem(
            label=app.state.classes[idx],
            confidence=round(float(probs[idx]), 4),
            percentage=f"{float(probs[idx]) * 100:.2f}%"
        ))

    return PredictionResponse(
        predicted_class=top_label,
        confidence=round(float(probs[top_idx]), 4),
        top_k=top_items,
        inference_time_ms=round(latency, 2),
        filename=file.filename
    )

## 4. Automated Testing with `TestClient`

We execute automated test cases covering valid predictions, invalid file types, corrupted images, and health checks.

In [5]:
client = TestClient(app)

# Helper to create dummy test image
def make_dummy_img():
    img = Image.new("RGB", (64, 64), color="red")
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

# Test 1: Health check
res = client.get("/health")
assert res.status_code == 200
assert res.json()["model_loaded"] is True
print(" Test 1 Passed: /health returns 200 and model_loaded=True")

# Test 2: Valid prediction
img_bytes = make_dummy_img()
res = client.post(
    "/predict",
    files={"file": ("test.png", img_bytes, "image/png")},
    data={"top_k": 3}
)
assert res.status_code == 200
data = res.json()
assert data["predicted_class"] in CLASSES
assert len(data["top_k"]) == 3
print(f" Test 2 Passed: /predict succeeded (class={data['predicted_class']}, conf={data['confidence']})")

# Test 3: Unsupported file extension
res = client.post("/predict", files={"file": ("notes.txt", b"plain text", "text/plain")})
assert res.status_code == 400
print(" Test 3 Passed: Rejects unsupported file extension with HTTP 400")

# Test 4: Corrupted file
res = client.post("/predict", files={"file": ("bad.png", b"not an image", "image/png")})
assert res.status_code == 400
print(" Test 4 Passed: Rejects corrupted image data with HTTP 400")

print("\n ALL FASTAPI INTEGRATION TESTS PASSED!")

 Test 1 Passed: /health returns 200 and model_loaded=True


 Test 2 Passed: /predict succeeded (class=dog, conf=0.4035)
 Test 3 Passed: Rejects unsupported file extension with HTTP 400
 Test 4 Passed: Rejects corrupted image data with HTTP 400

 ALL FASTAPI INTEGRATION TESTS PASSED!


## 5. Deployment Guidelines

To deploy this service in production:
1. **Run with Uvicorn:**
   ```bash
   uvicorn app:app --host 0.0.0.0 --port 8000 --workers 4
   ```
2. **Access Swagger UI:**
   Navigate to `http://localhost:8000/docs` for interactive schema documentation and API testing.
3. **Containerization:**
   Package the application in a lightweight Docker container with healthcheck monitoring.